# LuluCare 360 — Module 3 | `economist.py`
### The Economist — Resolution & Refund Economics Engine

**Project:** LuluCare 360 — AI Complaint-Resolution Co-Pilot  
**Module:** Module 3 — The Economist  
**File maps to:** `backend/modules/economist/economist.py`  
**Pipeline position:** Reader → Investigator → **[ECONOMIST]** → Voice

---

## What This Module Does
Given the Investigator's verdict, the Reader's output, and the customer profile, it decides:
- **`action`** — what resolution to offer (ACKNOWLEDGE / COUPON / WALLET_CREDIT / REFUND / ESCALATE)
- **`refund_type`** — logistics decision (PICKUP / KEEP_ITEM / NONE)
- **`coupon_percent`** — 0, 20, or 50
- **`wallet_credit`** — AED amount or 0
- **`escalate`** — bool, whether to hand to a human
- **`email_trigger`** — bool, fires ONLY for money actions

## What This Module NEVER Does
- ❌ Never writes a customer-facing sentence (that is the Voice's job)
- ❌ Never uses tone/frustration to grant trust or release money
- ❌ Never reads `_archetype` (leakage trap)
- ❌ Never renames output keys (contract is frozen)


## Cell 1 — Imports & Enum Setup

The Economist imports all string constants from `shared/enums.py`. Falls back to local definitions if running standalone (e.g. Colab). No bare string literals ever appear in branch logic.

In [1]:
import os
import re
import sys
import json

# ---------------------------------------------------------------------------
# Enum constants: import from shared source if present, else local fallback.
# WHY: integration-safety -- no bare string literals in branch logic.
# ---------------------------------------------------------------------------
_HERE = os.path.dirname(os.path.abspath('__file__'))
_ROOT = os.path.abspath(os.path.join(_HERE, '..', '..', '..'))
if _ROOT not in sys.path:
    sys.path.insert(0, _ROOT)

try:
    from shared.enums import (
        GENUINE, SUSPICIOUS, LIKELY_ABUSER,
        CONFIRMED, CONTRADICTED, UNVERIFIED,
        ACTION_ACKNOWLEDGE, ACTION_COUPON, ACTION_WALLET_CREDIT,
        ACTION_REFUND, ACTION_ESCALATE, ACTIONS, EMAIL_ACTIONS,
        REFUND_PICKUP, REFUND_KEEP_ITEM, REFUND_NONE, REFUND_TYPES,
        FRUST_LOW, FRUST_MEDIUM, FRUST_HIGH,
        TIER_GOLD, TIER_SILVER, TIER_PLATINUM,
        ISSUE_DAMAGED_DEFECTIVE,
        VALUE_HIGH, VALUE_MEDIUM, VALUE_LOW,
    )
    _ENUM_SOURCE = 'shared.enums'
except Exception:
    GENUINE, SUSPICIOUS, LIKELY_ABUSER = 'GENUINE', 'SUSPICIOUS', 'LIKELY_ABUSER'
    CONFIRMED, CONTRADICTED, UNVERIFIED = 'CONFIRMED', 'CONTRADICTED', 'UNVERIFIED'
    ACTION_ACKNOWLEDGE, ACTION_COUPON = 'ACKNOWLEDGE', 'COUPON'
    ACTION_WALLET_CREDIT, ACTION_REFUND, ACTION_ESCALATE = 'WALLET_CREDIT', 'REFUND', 'ESCALATE'
    ACTIONS = (ACTION_ACKNOWLEDGE, ACTION_COUPON, ACTION_WALLET_CREDIT, ACTION_REFUND, ACTION_ESCALATE)
    EMAIL_ACTIONS = (ACTION_COUPON, ACTION_REFUND, ACTION_WALLET_CREDIT)
    REFUND_PICKUP, REFUND_KEEP_ITEM, REFUND_NONE = 'PICKUP', 'KEEP_ITEM', 'NONE'
    REFUND_TYPES = (REFUND_PICKUP, REFUND_KEEP_ITEM, REFUND_NONE)
    FRUST_LOW, FRUST_MEDIUM, FRUST_HIGH = 'Low', 'Medium', 'High'
    TIER_GOLD, TIER_SILVER, TIER_PLATINUM = 'Gold', 'Silver', 'Platinum'
    ISSUE_DAMAGED_DEFECTIVE = 'Damaged_Defective'
    VALUE_HIGH, VALUE_MEDIUM, VALUE_LOW = 'HIGH', 'MEDIUM', 'LOW'
    _ENUM_SOURCE = 'local'

print(f'[economist] Enum source: {_ENUM_SOURCE}')
print('Imports OK ✅')


[economist] Enum source: local
Imports OK ✅


## Cell 2 — Tunable Policy Knobs

These are the numbers the team defends in the **Trust Memo**.  
All thresholds live in **one place** — never duplicated across branches.


In [2]:
# ---------------------------------------------------------------------------
# TUNABLE POLICY KNOBS (document and defend in the trust memo)
# ---------------------------------------------------------------------------
COUPON_STANDARD  = 20            # default goodwill coupon (%)
COUPON_GENEROUS  = 50            # high-frustration, high-value coupon (%)
ALLOWED_COUPONS  = (0, COUPON_STANDARD, COUPON_GENEROUS)

# Wallet-credit amounts by value band, in AED. TUNABLE policy choice.
WALLET_CREDIT_BY_BAND = {VALUE_LOW: 50, VALUE_MEDIUM: 100, VALUE_HIGH: 200}

NEW_ACCOUNT_MAX_MONTHS = 2       # 'newbie' threshold
HIGH_RESALE_THRESHOLD  = 2000    # costly resalable goods always worth recovering
ESCALATE_ORDER_VALUE   = 5000    # large-money threshold for escalation valve
LOW_CONFIDENCE         = 0.5     # Reader confidence floor for escalation valve

print('Policy knobs:')
print(f'  COUPON_STANDARD         = {COUPON_STANDARD}%')
print(f'  COUPON_GENEROUS         = {COUPON_GENEROUS}%')
print(f'  WALLET_CREDIT_BY_BAND   = {WALLET_CREDIT_BY_BAND}')
print(f'  NEW_ACCOUNT_MAX_MONTHS  = {NEW_ACCOUNT_MAX_MONTHS}')
print(f'  HIGH_RESALE_THRESHOLD   = AED {HIGH_RESALE_THRESHOLD}')
print(f'  ESCALATE_ORDER_VALUE    = AED {ESCALATE_ORDER_VALUE}')
print(f'  LOW_CONFIDENCE          = {LOW_CONFIDENCE}')


Policy knobs:
  COUPON_STANDARD         = 20%
  COUPON_GENEROUS         = 50%
  WALLET_CREDIT_BY_BAND   = {'LOW': 50, 'MEDIUM': 100, 'HIGH': 200}
  NEW_ACCOUNT_MAX_MONTHS  = 2
  HIGH_RESALE_THRESHOLD   = AED 2000
  ESCALATE_ORDER_VALUE    = AED 5000
  LOW_CONFIDENCE          = 0.5


## Cell 3 — CHECK 3: `value_band()` — Customer Value Classification

**Why it exists:** A generous gesture to a high-value customer is a *retention investment*, not a giveaway.  
Value never **grants** trust; it only **scales** remediation within an already-trusted path.

| Condition | Band |
|---|---|
| Platinum tier OR CLV >= 40,000 | `HIGH` |
| Gold/Silver tier OR CLV >= 12,000 | `MEDIUM` |
| Else | `LOW` |


In [3]:
def value_band(p):
    # Platinum or very high lifetime value -> HIGH
    if p['loyalty_tier'] == TIER_PLATINUM or p['clv_estimate'] >= 40000:
        return VALUE_HIGH
    # Gold/Silver or moderate CLV -> MEDIUM
    if p['loyalty_tier'] in (TIER_GOLD, TIER_SILVER) or p['clv_estimate'] >= 12000:
        return VALUE_MEDIUM
    # Everything else -> LOW
    return VALUE_LOW

# Quick verification
print('value_band tests:')
print(f"  Platinum, clv 117030  -> {value_band({'loyalty_tier':'Platinum','clv_estimate':117030})}  (expect HIGH)")
print(f"  Gold,     clv 16176   -> {value_band({'loyalty_tier':'Gold',     'clv_estimate':16176})}  (expect MEDIUM)")
print(f"  Bronze,   clv 4959    -> {value_band({'loyalty_tier':'Bronze',   'clv_estimate':4959})}   (expect LOW)")
print(f"  Silver,   clv 57134   -> {value_band({'loyalty_tier':'Silver',   'clv_estimate':57134})}  (expect HIGH, clv>=40000)")


value_band tests:
  Platinum, clv 117030  -> HIGH  (expect HIGH)
  Gold,     clv 16176   -> MEDIUM  (expect MEDIUM)
  Bronze,   clv 4959    -> LOW   (expect LOW)
  Silver,   clv 57134   -> HIGH  (expect HIGH, clv>=40000)


## Cell 4 — CHECK 5: `refund_logistics()` — Keep-vs-Pickup Tree

**Why it exists:** Pure economics — never pay to recover something you cannot resell,  
or something worth less than the freight.

```
if perishable/hygiene      -> KEEP_ITEM  (can't resell -> collecting is pure waste)
elif resale_value >= 2000  -> PICKUP     (costly resalable -> always recover)
elif freight > resale      -> KEEP_ITEM  (shipping back costs more than it's worth)
else                       -> PICKUP
```

**Verified against data:** 114 KEEP_ITEM (110 perishable + 4 by cost>resale), 106 PICKUP


In [4]:
def refund_logistics(p):
    # 1) Perishable / hygiene goods cannot be resold -> collecting is pure waste.
    if p['is_perishable_or_hygiene']:
        return REFUND_KEEP_ITEM
    # 2) Costly resalable goods always worth recovering.
    if p['resale_value'] >= HIGH_RESALE_THRESHOLD:
        return REFUND_PICKUP
    # 3) Weigh freight against recoverable value.
    if p['reverse_logistics_cost'] > p['resale_value']:
        return REFUND_KEEP_ITEM   # shipping it back costs more than it is worth
    return REFUND_PICKUP

# Verify all branches
perishable  = {'is_perishable_or_hygiene': True,  'resale_value': 0,     'reverse_logistics_cost': 108}
electronics = {'is_perishable_or_hygiene': False, 'resale_value': 34890, 'reverse_logistics_cost': 78}
cheap_item  = {'is_perishable_or_hygiene': False, 'resale_value': 164,   'reverse_logistics_cost': 180}
normal_item = {'is_perishable_or_hygiene': False, 'resale_value': 200,   'reverse_logistics_cost': 121}

print('refund_logistics tests:')
print(f'  C0005 perishable        -> {refund_logistics(perishable)}  (expect KEEP_ITEM)')
print(f'  C0013 Electronics       -> {refund_logistics(electronics)}  (expect PICKUP)')
print(f'  C0032 resale<freight    -> {refund_logistics(cheap_item)}  (expect KEEP_ITEM)')
print(f'  C0006 resale>freight    -> {refund_logistics(normal_item)}  (expect PICKUP)')


refund_logistics tests:
  C0005 perishable        -> KEEP_ITEM  (expect KEEP_ITEM)
  C0013 Electronics       -> PICKUP  (expect PICKUP)
  C0032 resale<freight    -> KEEP_ITEM  (expect KEEP_ITEM)
  C0006 resale>freight    -> PICKUP  (expect PICKUP)


## Cell 5 — CHECK 6: `should_escalate()` — Escalation Valve

**Why it exists:** Escalate ONLY when stakes are high AND the system is unsure.  
Over-escalating re-buries the team and destroys the automation business case.

| Condition | Escalate? |
|---|---|
| Proposed REFUND + SUSPICIOUS + order > AED 5,000 | Yes |
| Reader confidence < 0.5 + value band HIGH | Yes |
| Anything else | No |


In [5]:
def should_escalate(verdict, reader, p, proposed_refund):
    # Large money + uncertain trust -> warrants a human's eyes.
    if proposed_refund and verdict['genuineness'] == SUSPICIOUS and p['order_value'] > ESCALATE_ORDER_VALUE:
        return True
    # Unsure about a valuable customer -> risks a costly wrong call.
    if reader['confidence'] < LOW_CONFIDENCE and value_band(p) == VALUE_HIGH:
        return True
    return False

v_sus  = {'genuineness': SUSPICIOUS, 'claim_status': UNVERIFIED, 'reason': ''}
v_gen  = {'genuineness': GENUINE,    'claim_status': UNVERIFIED, 'reason': ''}
r_low  = {'issue_type': 'Delivery',  'frustration': FRUST_HIGH,  'confidence': 0.45}
r_high = {'issue_type': 'Delivery',  'frustration': FRUST_HIGH,  'confidence': 0.90}
p_big  = {'loyalty_tier': 'Platinum', 'clv_estimate': 114588, 'order_value': 24392,
           'is_perishable_or_hygiene': False, 'resale_value': 14635, 'reverse_logistics_cost': 212}
p_sml  = {'loyalty_tier': 'Bronze',   'clv_estimate': 4959,   'order_value': 224,
           'is_perishable_or_hygiene': True,  'resale_value': 0,     'reverse_logistics_cost': 118}

print('should_escalate tests:')
print(f'  C0059 SUSPICIOUS + order 24392 + REFUND -> {should_escalate(v_sus, r_high, p_big, True)}  (expect True)')
print(f'  C0059 conf 0.45 + HIGH value            -> {should_escalate(v_sus, r_low,  p_big, False)}  (expect True)')
print(f'  C0020 GENUINE  + small order + HIGH conf -> {should_escalate(v_gen, r_high, p_sml, False)}  (expect False)')


should_escalate tests:
  C0059 SUSPICIOUS + order 24392 + REFUND -> True  (expect True)
  C0059 conf 0.45 + HIGH value            -> True  (expect True)
  C0020 GENUINE  + small order + HIGH conf -> False  (expect False)


## Cell 6 — Helper Functions for CONFIRMED-Promise Branch

`_extract_percent()` — pulls a logged coupon % from the agent's notes  
`_confirmed_remedy()` — decides what to honour based on what was logged


In [6]:
def _extract_percent(text):
    """Pull a logged coupon percentage out of the verdict reason/notes, if any."""
    m = re.search(r'(\d{1,3})\s*%', text)
    return int(m.group(1)) if m else None


def _confirmed_remedy(verdict):
    """
    Honour exactly what an agent LOGGED. Default to a full REFUND; only switch
    to COUPON when the logged note explicitly mentions a coupon/discount/%.
    A logged promise is OUR record (unfakeable by customer).
    """
    reason = str(verdict.get('reason', '')).lower()
    pct = _extract_percent(reason)
    mentions_coupon = ('coupon' in reason) or ('discount' in reason) or (pct is not None)
    if mentions_coupon:
        if pct in (COUPON_STANDARD, COUPON_GENEROUS):
            cp = pct
        elif (pct or 0) >= 35:
            cp = COUPON_GENEROUS
        else:
            cp = COUPON_STANDARD
        return ACTION_COUPON, cp, 0
    return ACTION_REFUND, 0, 0

print('_confirmed_remedy tests:')
print(f"  notes='committed to a refund'    -> {_confirmed_remedy({'reason':'Prior agent committed to a refund, pending processing.'})}")
print(f"  notes='promised 20% coupon'      -> {_confirmed_remedy({'reason':'Agent promised 20% coupon on previous call, not yet issued.'})}")
print(f"  notes='assured replacement'      -> {_confirmed_remedy({'reason':'Customer was assured replacement on last contact.'})}")


_confirmed_remedy tests:
  notes='committed to a refund'    -> ('REFUND', 0, 0)
  notes='promised 20% coupon'      -> ('COUPON', 20, 0)
  notes='assured replacement'      -> ('REFUND', 0, 0)


## Cell 7 — CHECK 4: `choose_action()` — The 8-Rule Remediation Tier Table

**Anti-abuse precedence is FIXED — evaluate top-to-bottom, first match wins.**

| # | Condition | Action |
|---|---|---|
| 1 | `claim_status == CONTRADICTED` | ACKNOWLEDGE (no payout) |
| 2 | `claim_status == CONFIRMED` | Honour logged promise |
| 3 | `genuineness == LIKELY_ABUSER` | ACKNOWLEDGE (no payout) |
| 4 | First purchase OR account <= 2 months | COUPON 20% (capped) |
| 5 | GENUINE + Damaged_Defective | REFUND |
| 6 | GENUINE + High frustration + HIGH value | COUPON 50% |
| 7 | GENUINE + Medium frustration | COUPON 20% or WALLET_CREDIT |
| 8 | Everything else (GENUINE, Low, routine) | ACKNOWLEDGE |


In [7]:
def choose_action(verdict, reader, p):
    g     = verdict['genuineness']
    claim = verdict['claim_status']
    issue = reader['issue_type']
    frust = reader['frustration']
    band  = value_band(p)

    # 1) CONTRADICTED -- our notes disprove the claim. No payout, no email.
    if claim == CONTRADICTED:
        return ACTION_ACKNOWLEDGE, 0, 0, 'claim CONTRADICTED by our records -> respond from record, no payout'

    # 2) CONFIRMED -- an agent LOGGED a promise. Honour exactly what was logged.
    if claim == CONFIRMED:
        action, cp, wc = _confirmed_remedy(verdict)
        if g == LIKELY_ABUSER:
            # C0006 conflict: honour our record, but cap to promised scope only.
            note = ('CONFIRMED logged promise honoured; genuineness=LIKELY_ABUSER -> '
                    'ABUSE_FLAG: capped to promised remedy, no extra remediation, normal logistics')
            return action, cp, wc, note
        return action, cp, wc, 'CONFIRMED logged promise -> honour exactly what was logged'

    # 3) LIKELY_ABUSER -- history (not tone) shows gaming. Anger is not evidence.
    if g == LIKELY_ABUSER:
        return ACTION_ACKNOWLEDGE, 0, 0, 'genuineness LIKELY_ABUSER -> acknowledge only, abuse never pays'

    # 4) New account / first purchase -- generous but capped.
    if p['is_first_purchase'] or p['account_age_months'] <= NEW_ACCOUNT_MAX_MONTHS:
        return ACTION_COUPON, COUPON_STANDARD, 0, 'new/first-purchase account -> generous-but-capped coupon, verify'

    # 5) GENUINE + clear product failure -> REFUND.
    #    Placed BEFORE frustration rules so a calm genuine defect is never downgraded.
    if g == GENUINE and issue == ISSUE_DAMAGED_DEFECTIVE:
        return ACTION_REFUND, 0, 0, 'GENUINE + Damaged_Defective -> refund (logistics by economics)'

    # 6) GENUINE + High frustration + HIGH value -> generous coupon.
    if g == GENUINE and frust == FRUST_HIGH and band == VALUE_HIGH:
        return ACTION_COUPON, COUPON_GENEROUS, 0, 'GENUINE + High frustration + HIGH value -> generous coupon'

    # 7) GENUINE + Medium frustration -> standard goodwill, scaled by value.
    if g == GENUINE and frust == FRUST_MEDIUM:
        if band == VALUE_LOW:
            return ACTION_WALLET_CREDIT, 0, WALLET_CREDIT_BY_BAND[VALUE_LOW], 'GENUINE + Medium + LOW value -> flat wallet credit'
        return ACTION_COUPON, COUPON_STANDARD, 0, 'GENUINE + Medium frustration -> standard coupon'

    # 8) Everything else (GENUINE, Low, routine) -> acknowledge.
    return ACTION_ACKNOWLEDGE, 0, 0, 'GENUINE routine / Low frustration -> acknowledge or small gesture'

print('choose_action loaded ✅  (orchestrated in decide() below)')


choose_action loaded ✅  (orchestrated in decide() below)


## Cell 8 — CHECK 7: `decide()` — Main Orchestrator

**This is the function the rest of the team calls.**  
Assembles the full, JSON-safe contract dict:

```
choose_action() -> refund_logistics() -> should_escalate() -> email_trigger -> cast all types
```

**Critical:** `email_trigger` is recomputed **AFTER** escalation override,  
so an escalated case can never accidentally leak an email.


In [8]:
def decide(verdict, reader, p):
    # a) pick the action + amounts from the tier table
    action, coupon_percent, wallet_credit, note = choose_action(verdict, reader, p)

    # b) logistics only matter for an actual refund; otherwise NONE
    refund_type = refund_logistics(p) if action == ACTION_REFUND else REFUND_NONE

    # c) escalation valve -- evaluated against the *proposed* action
    escalate = should_escalate(verdict, reader, p, proposed_refund=(action == ACTION_REFUND))
    if escalate:
        action = ACTION_ESCALATE
        note   = note + ' | OVERRIDE: escalated to human (high stakes + low certainty)'

    # d) email fires for money actions ONLY, recomputed AFTER escalation
    email_trigger = action in EMAIL_ACTIONS

    # e) cast EVERYTHING to native Python types (pandas gives NumPy scalars)
    return {
        'action':         str(action),
        'refund_type':    str(refund_type),
        'coupon_percent': int(coupon_percent),
        'wallet_credit':  int(wallet_credit),
        'escalate':       bool(escalate),
        'email_trigger':  bool(email_trigger),
        'reason':         str(note),
    }

print('decide() loaded ✅')


decide() loaded ✅


## Cell 9 — Demo Runner (5 Key Cases)

| Case | Customer | Scenario | Expected |
|---|---|---|---|
| 1 | C0005 | Perishable defect | REFUND + KEEP_ITEM |
| 3 | C0001 | Serial abuser, furious, HIGH CLV | ACKNOWLEDGE (abuse never pays) |
| 6 | C0032 | Cheap item, freight > resale | REFUND + KEEP_ITEM (economics) |
| 10 | C0018 | Platinum, CALM, genuine defect | REFUND + PICKUP (trap case) |
| 11 | C0006 | Abuser WITH logged promise | REFUND + PICKUP + ABUSE_FLAG |


In [9]:
demos = [
    ('Case 1  C0005 perishable defect (REFUND+KEEP_ITEM)',
     {'genuineness': GENUINE,       'claim_status': UNVERIFIED, 'reason': 'clean history'},
     {'issue_type': ISSUE_DAMAGED_DEFECTIVE, 'frustration': FRUST_HIGH, 'confidence': 0.91},
     {'loyalty_tier': 'Gold',     'clv_estimate': 37040,  'order_value': 394,
      'is_first_purchase': False, 'account_age_months': 27,
      'is_perishable_or_hygiene': True,  'resale_value': 0,     'reverse_logistics_cost': 108}),

    ('Case 3  C0001 serial abuser, furious, HIGH clv (ACKNOWLEDGE no email)',
     {'genuineness': LIKELY_ABUSER, 'claim_status': UNVERIFIED, 'reason': 'ratio 0.833'},
     {'issue_type': 'Delivery',     'frustration': FRUST_HIGH,  'confidence': 0.90},
     {'loyalty_tier': 'Silver',   'clv_estimate': 159720, 'order_value': 9164,
      'is_first_purchase': False, 'account_age_months': 1,
      'is_perishable_or_hygiene': False, 'resale_value': 5956,  'reverse_logistics_cost': 101}),

    ('Case 6  C0032 cheap item, freight > resale (REFUND+KEEP_ITEM economics)',
     {'genuineness': GENUINE,       'claim_status': UNVERIFIED, 'reason': 'clean history'},
     {'issue_type': ISSUE_DAMAGED_DEFECTIVE, 'frustration': FRUST_MEDIUM, 'confidence': 0.88},
     {'loyalty_tier': 'Silver',   'clv_estimate': 57134,  'order_value': 410,
      'is_first_purchase': False, 'account_age_months': 49,
      'is_perishable_or_hygiene': False, 'resale_value': 164,   'reverse_logistics_cost': 180}),

    ('Case 10 C0018 Platinum CALM genuine defect (must still REFUND+PICKUP)',
     {'genuineness': GENUINE,       'claim_status': UNVERIFIED, 'reason': 'clean history'},
     {'issue_type': ISSUE_DAMAGED_DEFECTIVE, 'frustration': FRUST_LOW, 'confidence': 0.93},
     {'loyalty_tier': 'Platinum', 'clv_estimate': 117030, 'order_value': 66383,
      'is_first_purchase': False, 'account_age_months': 20,
      'is_perishable_or_hygiene': False, 'resale_value': 43148, 'reverse_logistics_cost': 121}),

    ('Case 11 C0006 abuser WITH logged promise (honour PICKUP ABUSE_FLAG)',
     {'genuineness': LIKELY_ABUSER, 'claim_status': CONFIRMED,
      'reason': 'Customer was assured replacement on last contact.'},
     {'issue_type': 'Refund_Return', 'frustration': FRUST_HIGH, 'confidence': 0.87},
     {'loyalty_tier': 'Silver',   'clv_estimate': 25926,  'order_value': 445,
      'is_first_purchase': False, 'account_age_months': 14,
      'is_perishable_or_hygiene': False, 'resale_value': 200,   'reverse_logistics_cost': 121}),
]

print(f'[economist] enum source: {_ENUM_SOURCE}\n')
print('=' * 72)
for title, v, r, p in demos:
    result = decide(v, r, p)
    print(f'\n{title}')
    print(json.dumps(result, indent=2))
    print('-' * 72)


[economist] enum source: local


Case 1  C0005 perishable defect (REFUND+KEEP_ITEM)
{
  "action": "REFUND",
  "refund_type": "KEEP_ITEM",
  "coupon_percent": 0,
  "wallet_credit": 0,
  "escalate": false,
  "email_trigger": true,
  "reason": "GENUINE + Damaged_Defective -> refund (logistics by economics)"
}
------------------------------------------------------------------------

Case 3  C0001 serial abuser, furious, HIGH clv (ACKNOWLEDGE no email)
{
  "action": "ACKNOWLEDGE",
  "refund_type": "NONE",
  "coupon_percent": 0,
  "wallet_credit": 0,
  "escalate": false,
  "email_trigger": false,
  "reason": "genuineness LIKELY_ABUSER -> acknowledge only, abuse never pays"
}
------------------------------------------------------------------------

Case 6  C0032 cheap item, freight > resale (REFUND+KEEP_ITEM economics)
{
  "action": "REFUND",
  "refund_type": "KEEP_ITEM",
  "coupon_percent": 0,
  "wallet_credit": 0,
  "escalate": false,
  "email_trigger": true,
  "reason": "GENUINE + Damaged_